In [1]:
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig
import json
from datasets import Dataset

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 512

### Load Dataset and Specify Train and Eval Partitions

In [3]:
with open("train.json", "r") as f:
    train_data = json.load(f)
    train_dataset = Dataset.from_list(train_data)

with open("validation.json", "r") as f:
    validation_data = json.load(f)
    validation_dataset = Dataset.from_list(validation_data)


### Define Data Processing Function

In [4]:
def sample_formatting_function(row):
    """Format one training row for schema-linking SFT."""
    fname = row["db_id"].replace(" ", "_").replace("/", "_") + ".json"
    with open(f"./schemas/{fname}") as f:
        s = json.load(f)

    schema = {t: [] for t in s["table_names_original"]}
    for tidx, cname in s["column_names_original"]:
        if tidx == -1:
            continue
        schema[s["table_names_original"][tidx]].append(cname)

    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists."
    )

    prompt = (
        f"Database schema: {schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return JSON only in this format: {\"TableName\": [\"col1\", \"col2\"], ...}"
    )

    completion_json = json.dumps(row["schema_links"], ensure_ascii=False)

    return {
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        "completion": [
            {"role": "assistant", "content": completion_json}
        ]
    }

### Initialize Experiment

In [5]:
# Every experiment instance must be uniquely named
experiment = Experiment(experiment_name="exper1", mode="fit")

Experiment exper1 is currently running. Returning the same experiment object.


### Define Multi-Config Knobs for Model, LoRA, and SFT Trainer using RapidFire AI Wrapper APIs

In [6]:
# 2 LoRA PEFT configs lite with different adapter capacities
peft_configs_lite = List([
    RFLoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["q_proj", "v_proj"],  # Standard transformer naming
        bias="none"
    ),
    RFLoraConfig(
        r=32,
        lora_alpha=64,
        lora_dropout=0.1,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Standard naming
        bias="none"
    )
])

# 2 base models x 2 peft configs = 4 combinations in total
config_set_lite = List([
    RFModelConfig(
        model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",  # 1.1B model
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=1e-3,  # Higher LR for very small model
            lr_scheduler_type="linear",
            per_device_train_batch_size=4,
            per_device_eval_batch_size=4,
            max_steps=128,
            gradient_accumulation_steps=1,   # No accumulation needed
            logging_steps=2,
            eval_strategy="steps",
            eval_steps=4,
            bf16=True,
        ),
        model_type="causal_lm",
        model_kwargs={"device_map": "auto", "torch_dtype": "auto", "use_cache": False},
        formatting_func=sample_formatting_function,
        generation_config={
            "max_new_tokens": 256,
            "temperature": 0.8,  # Higher temp for tiny model
            "top_p": 0.9,
            "top_k": 30,         # Reduced top_k
            "repetition_penalty": 1.05,
        }
    ),
    RFModelConfig(
        model_name=MODEL_NAME,
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=1e-4,  # Higher LR for very small model
            lr_scheduler_type="linear",
            per_device_train_batch_size=4,  # Even larger batch size
            per_device_eval_batch_size=4,
            max_steps=128,
            gradient_accumulation_steps=1,   # No accumulation needed
            logging_steps=2,
            eval_strategy="steps",
            eval_steps=4,
            bf16=True,
        ),
        model_type="causal_lm",
        model_kwargs={"device_map": "auto", "torch_dtype": "auto", "use_cache": False},
        formatting_func=sample_formatting_function,
        generation_config={
            "max_new_tokens": 256,
            "temperature": 0.8,  # Higher temp for tiny model
            "top_p": 0.9,
            "top_k": 30,         # Reduced top_k
            "repetition_penalty": 1.05,
        }
    )
])


#### Define Model Creation Function for All Model Types Across Configs

In [7]:

def sample_create_model(model_config): 
     """Function to create model object for any given config; must return tuple of (model, tokenizer)"""
     from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForMaskedLM

     model_name = model_config["model_name"]
     model_type = model_config["model_type"]
     model_kwargs = model_config["model_kwargs"]
 
     if model_type == "causal_lm":
          model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "seq2seq_lm":
          model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "masked_lm":
          model = AutoModelForMaskedLM.from_pretrained(model_name, **model_kwargs)
     elif model_type == "custom":
          # Handle custom model loading logic, e.g., loading your own checkpoints
          # model = ... 
          pass
     else:
          # Default to causal LM
          model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
      
     tokenizer = AutoTokenizer.from_pretrained(model_name)
      
     return (model,tokenizer)


#### Generate Config Group

In [8]:
# Simple grid search across all sets of config knob values = 4 combinations in total
config_group = RFGridSearch(
    configs=config_set_lite,
    trainer_type="SFT"
)

### Run Multi-Config Training

In [9]:
# Launch training of all configs in the config_group with swap granularity of 4 chunks
experiment.run_fit(config_group, sample_create_model, train_dataset, validation_dataset, num_chunks=4, seed=42)

Started 1 worker processes successfully
Created workers


Run 6 has failed: name 'load_schema_as_dict' is not definedTraceback (most recent call last):
  File "/home/sjrao/.local/lib/python3.13/site-packages/rapidfireai/fit/backend/worker.py", line 676, in serve_forever
    self.run_fit(
    ~~~~~~~~~~~~^
        run_id, chunk_id, multi_worker_details, create_model_fn
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/sjrao/.local/lib/python3.13/site-packages/rapidfireai/fit/backend/worker.py", line 354, in run_fit
    trainer_instance, base_model_name = create_trainer_instance(
                                        ~~~~~~~~~~~~~~~~~~~~~~~^
        trainer_config,
        ^^^^^^^^^^^^^^^
    ...<4 lines>...
        use_fsdp=use_fsdp,
        ^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/sjrao/.local/lib/python3.13/site-packages/rapidfireai/fit/ml/trainer.py", line 425, in create_trainer_instance
    _prepare_trainer_kwargs(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        model_instance,
        ^^^^^^^^^^^^^^^
   

Run 7 has failed: name 'load_schema_as_dict' is not definedTraceback (most recent call last):
  File "/home/sjrao/.local/lib/python3.13/site-packages/rapidfireai/fit/backend/worker.py", line 676, in serve_forever
    self.run_fit(
    ~~~~~~~~~~~~^
        run_id, chunk_id, multi_worker_details, create_model_fn
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/sjrao/.local/lib/python3.13/site-packages/rapidfireai/fit/backend/worker.py", line 354, in run_fit
    trainer_instance, base_model_name = create_trainer_instance(
                                        ~~~~~~~~~~~~~~~~~~~~~~~^
        trainer_config,
        ^^^^^^^^^^^^^^^
    ...<4 lines>...
        use_fsdp=use_fsdp,
        ^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/sjrao/.local/lib/python3.13/site-packages/rapidfireai/fit/ml/trainer.py", line 425, in create_trainer_instance
    _prepare_trainer_kwargs(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        model_instance,
        ^^^^^^^^^^^^^^^
   

Run 8 has failed: name 'load_schema_as_dict' is not definedTraceback (most recent call last):
  File "/home/sjrao/.local/lib/python3.13/site-packages/rapidfireai/fit/backend/worker.py", line 676, in serve_forever
    self.run_fit(
    ~~~~~~~~~~~~^
        run_id, chunk_id, multi_worker_details, create_model_fn
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/sjrao/.local/lib/python3.13/site-packages/rapidfireai/fit/backend/worker.py", line 354, in run_fit
    trainer_instance, base_model_name = create_trainer_instance(
                                        ~~~~~~~~~~~~~~~~~~~~~~~^
        trainer_config,
        ^^^^^^^^^^^^^^^
    ...<4 lines>...
        use_fsdp=use_fsdp,
        ^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/sjrao/.local/lib/python3.13/site-packages/rapidfireai/fit/ml/trainer.py", line 425, in create_trainer_instance
    _prepare_trainer_kwargs(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        model_instance,
        ^^^^^^^^^^^^^^^
   

Run 5 has failed: name 'load_schema_as_dict' is not definedTraceback (most recent call last):
  File "/home/sjrao/.local/lib/python3.13/site-packages/rapidfireai/fit/backend/worker.py", line 676, in serve_forever
    self.run_fit(
    ~~~~~~~~~~~~^
        run_id, chunk_id, multi_worker_details, create_model_fn
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/sjrao/.local/lib/python3.13/site-packages/rapidfireai/fit/backend/worker.py", line 354, in run_fit
    trainer_instance, base_model_name = create_trainer_instance(
                                        ~~~~~~~~~~~~~~~~~~~~~~~^
        trainer_config,
        ^^^^^^^^^^^^^^^
    ...<4 lines>...
        use_fsdp=use_fsdp,
        ^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/sjrao/.local/lib/python3.13/site-packages/rapidfireai/fit/ml/trainer.py", line 425, in create_trainer_instance
    _prepare_trainer_kwargs(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        model_instance,
        ^^^^^^^^^^^^^^^
   

### End Current Experiment

In [ ]:
experiment.end()

<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Thanks for trying RapidFire AI! ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
</div>